# 06 — QAT TensorRT Inference (Test Set)

Evaluate QAT (INT8 via ModelOpt) checkpoints using **TensorRT** instead of
PyTorch fake-quantization.

Pipeline per input bit-depth:
1. Load QAT checkpoint → restore ModelOpt quant graph → export QDQ ONNX
2. Build TRT INT8 engine from QDQ ONNX
3. Run TRT inference on test set (`/home/pf4636/imagenet2`)

In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path("..").resolve()
PYFILES = PROJECT_ROOT / "pyfiles"
QAT_MOD = PYFILES / "qat_modelopt"
for p in [str(PYFILES), str(QAT_MOD)]:
    if p not in sys.path:
        sys.path.insert(0, p)

In [2]:
import json
import time
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
import modelopt.torch.opt as mto

from dataclasses import replace
from torch.utils.data import DataLoader

from src.config import ExperimentConfig, set_seed
from src.data import build_imagenet_dataset
from trt.trt_builder import build_engine
from trt.trt_infer import trt_evaluate
from utils.onnx_exporter import ONNXExporter
from quantize import get_model, restore_modelopt_state

pd.set_option("display.max_columns", 200)
pd.set_option("display.float_format", "{:.3f}".format)
print("torch:", torch.__version__, "| cuda:", torch.cuda.is_available())

torch: 2.10.0+cu130 | cuda: True


In [3]:
QAT_CKPT_ROOT  = PROJECT_ROOT / "checkpoints" / "qat"
FP32_CKPT_ROOT = PROJECT_ROOT / "checkpoints"
ONNX_DIR       = PROJECT_ROOT / "onnx" / "qat"
ENGINE_DIR     = PROJECT_ROOT / "engines" / "qat"
TEST_ROOT      = "/home/pf4636/imagenet2"
INPUT_BITS     = [8, 4, 2, 1]
DEVICE         = torch.device("cuda" if torch.cuda.is_available() else "cpu")
SKIP_EXISTING  = True

checkpoints = {}
for bits in INPUT_BITS:
    run_name = f"int8_in{bits}b"
    ckpt_dir = QAT_CKPT_ROOT / run_name / "seed_42"
    fp32_ckpt = FP32_CKPT_ROOT / f"fp32_{bits}bit" / "seed_42" / "best.pth"
    checkpoints[bits] = {
        "weights": ckpt_dir / "qat_modelopt_best.pth",
        "mostate": ckpt_dir / "qat_modelopt_best_mostate.pt",
        "fp32_ckpt": fp32_ckpt,
    }
    assert checkpoints[bits]["weights"].exists(), f"Missing: {checkpoints[bits]['weights']}"
    assert checkpoints[bits]["mostate"].exists(), f"Missing: {checkpoints[bits]['mostate']}"
    assert fp32_ckpt.exists(), f"Missing: {fp32_ckpt}"

print("QAT checkpoints:")
for bits, paths in checkpoints.items():
    print(f"  in{bits}b: {paths['weights'].name}  (FP32 base: {paths['fp32_ckpt']})")
print(f"Test set:   {TEST_ROOT}")
print(f"ONNX dir:   {ONNX_DIR}")
print(f"Engine dir: {ENGINE_DIR}")

QAT checkpoints:
  in8b: qat_modelopt_best.pth  (FP32 base: /home/pf4636/code/resnet/quantized_resnets/checkpoints/fp32_8bit/seed_42/best.pth)
  in4b: qat_modelopt_best.pth  (FP32 base: /home/pf4636/code/resnet/quantized_resnets/checkpoints/fp32_4bit/seed_42/best.pth)
  in2b: qat_modelopt_best.pth  (FP32 base: /home/pf4636/code/resnet/quantized_resnets/checkpoints/fp32_2bit/seed_42/best.pth)
  in1b: qat_modelopt_best.pth  (FP32 base: /home/pf4636/code/resnet/quantized_resnets/checkpoints/fp32_1bit/seed_42/best.pth)
Test set:   /home/pf4636/imagenet2
ONNX dir:   /home/pf4636/code/resnet/quantized_resnets/onnx/qat
Engine dir: /home/pf4636/code/resnet/quantized_resnets/engines/qat


## Step 1 — Export QAT Models to QDQ ONNX

Load each QAT checkpoint (with ModelOpt fake-quant nodes), then `torch.onnx.export`.
The Q/DQ nodes are preserved in the ONNX graph, giving TensorRT the quantization
scales it needs — no runtime calibrator required.

In [4]:
def load_qat_model(bits: int) -> nn.Module:
    """Build base model from matching FP32 checkpoint, restore modelopt quant graph, load QAT weights."""
    fp32_ckpt = str(checkpoints[bits]["fp32_ckpt"])
    model = get_model(fp32_ckpt, num_classes=100)
    restore_modelopt_state(model, str(checkpoints[bits]["mostate"]))
    ckpt = torch.load(checkpoints[bits]["weights"], map_location="cpu")
    state = ckpt["model"] if "model" in ckpt else ckpt
    model.load_state_dict(state)
    model.eval()
    return model

def export_qat_onnx(model: nn.Module, onnx_path: Path):
    """Export ModelOpt QAT model to ONNX with Q/DQ nodes using legacy exporter."""
    onnx_path.parent.mkdir(parents=True, exist_ok=True)
    model.eval().cpu()
    dummy = torch.randn(1, 3, 224, 224)
    with torch.no_grad():
        torch.onnx.export(
            model, (dummy,), str(onnx_path),
            input_names=["images"],
            output_names=["logits"],
            opset_version=17,
            dynamic_axes={"images": {0: "batch"}, "logits": {0: "batch"}},
            export_params=True,
            do_constant_folding=True,
            dynamo=False,
        )
    print(f"  Saved ({onnx_path.stat().st_size / 1e6:.1f} MB)")

for bits in INPUT_BITS:
    run_name = f"int8_in{bits}b"
    onnx_path = ONNX_DIR / run_name / "resnet18_qat_int8_qdq.onnx"

    if SKIP_EXISTING and onnx_path.exists():
        print(f"  in{bits}b: ONNX exists, skipping → {onnx_path}")
        continue

    print(f"\n--- Exporting QAT ONNX for input_bits={bits} ---")
    model = load_qat_model(bits)
    export_qat_onnx(model, onnx_path)

    del model
    torch.cuda.empty_cache()

print("\nAll QAT ONNX exports ready.")

  in8b: ONNX exists, skipping → /home/pf4636/code/resnet/quantized_resnets/onnx/qat/int8_in8b/resnet18_qat_int8_qdq.onnx
  in4b: ONNX exists, skipping → /home/pf4636/code/resnet/quantized_resnets/onnx/qat/int8_in4b/resnet18_qat_int8_qdq.onnx
  in2b: ONNX exists, skipping → /home/pf4636/code/resnet/quantized_resnets/onnx/qat/int8_in2b/resnet18_qat_int8_qdq.onnx
  in1b: ONNX exists, skipping → /home/pf4636/code/resnet/quantized_resnets/onnx/qat/int8_in1b/resnet18_qat_int8_qdq.onnx

All QAT ONNX exports ready.


In [5]:
import onnx

print("Q/DQ node counts per QAT ONNX:")
for bits in INPUT_BITS:
    path = ONNX_DIR / f"int8_in{bits}b" / "resnet18_qat_int8_qdq.onnx"
    if not path.exists():
        print(f"  in{bits}b: NOT FOUND")
        continue
    m = onnx.load(str(path), load_external_data=False)
    ops = [n.op_type for n in m.graph.node]
    batch_dim = m.graph.input[0].type.tensor_type.shape.dim[0].dim_value
    batch_str = "dynamic" if batch_dim == 0 else str(batch_dim)
    print(f"  in{bits}b: Q={ops.count('QuantizeLinear'):3d}  DQ={ops.count('DequantizeLinear'):3d}  batch={batch_str}  size={path.stat().st_size/1e6:.1f}MB")

Q/DQ node counts per QAT ONNX:
  in8b: Q= 44  DQ= 44  batch=dynamic  size=45.0MB
  in4b: Q= 44  DQ= 44  batch=dynamic  size=45.0MB
  in2b: Q= 44  DQ= 44  batch=dynamic  size=45.0MB
  in1b: Q= 44  DQ= 44  batch=dynamic  size=45.0MB


## Step 2 — Build TRT INT8 Engines & Run Inference

In [6]:
OUT_DIR = Path("../runs/qat_trt_test").resolve()
OUT_DIR.mkdir(parents=True, exist_ok=True)

criterion = nn.CrossEntropyLoss()
records = []

for bits in INPUT_BITS:
    run_id = f"resnet18_qat_int8_in{bits}b_trt_bs1"
    result_path = OUT_DIR / run_id / "result.json"

    if SKIP_EXISTING and result_path.exists():
        print(f"  in{bits}b: exists, skipping")
        with open(result_path) as f:
            records.append(json.load(f))
        continue

    print(f"\n{'='*60}")
    print(f"QAT INT8 TRT — input_bits={bits}")
    print(f"{'='*60}")

    onnx_path = ONNX_DIR / f"int8_in{bits}b" / "resnet18_qat_int8_qdq.onnx"
    engine_path = ENGINE_DIR / f"{run_id}.engine"

    # Build engine
    if not engine_path.exists():
        print(f"[Step 2] Building TRT INT8 engine from {onnx_path.name} ...")
        build_engine(
            onnx_path=onnx_path,
            engine_path=engine_path,
            precision="int8",
            batch_size=1,
            workspace_mb=2048,
        )
    else:
        print(f"[Step 2] Engine exists, skipping: {engine_path.name}")

    # Run TRT inference
    print(f"[Step 3] Running TRT inference ...")
    cfg = ExperimentConfig(
        backend="tensorrt",
        device="cuda",
        batch_size=1,
        seed=42,
        num_eval_batches=500,
        input_quant_bits=bits,
        model_precision="int8",
    )
    set_seed(cfg)

    test_cfg = replace(cfg, imagenet_path=TEST_ROOT)
    test_dataset = build_imagenet_dataset(test_cfg, split="val")
    test_loader = DataLoader(
        test_dataset,
        batch_size=cfg.batch_size,
        shuffle=False,
        num_workers=cfg.num_workers,
        pin_memory=True,
        drop_last=True,
    )

    t0 = time.perf_counter()
    tracker = trt_evaluate(engine_path, cfg, test_loader, criterion)
    elapsed = time.perf_counter() - t0

    r = tracker.summary()
    print(f"  top1={r['top1_acc']:.2f}%  top5={r['top5_acc']:.2f}%  infer={r['infer_ms_avg']:.3f}ms")

    payload = {
        "status": "ok",
        "run_id": run_id,
        "system": cfg.stamp(),
        "config": cfg.to_dict(),
        "config_extra": {
            "qat_precision": "int8",
            "input_quant_bits": bits,
            "seed": 42,
            "backend": "tensorrt",
        },
        "results": r,
        "error": None,
        "total_eval_time_sec": round(elapsed, 3),
    }

    save_path = OUT_DIR / run_id / "result.json"
    save_path.parent.mkdir(parents=True, exist_ok=True)
    with open(save_path, "w") as f:
        json.dump(payload, f, indent=2, sort_keys=True)

    records.append(payload)
    torch.cuda.empty_cache()

print(f"\n{len(records)} runs complete.")

  in8b: exists, skipping
  in4b: exists, skipping
  in2b: exists, skipping
  in1b: exists, skipping

4 runs complete.


## Results Summary

In [7]:
rows = []
for rec in records:
    r = rec["results"]
    extra = rec.get("config_extra", rec.get("config", {}))
    bits = extra.get("input_quant_bits", rec["config"].get("input_quant_bits", None))
    rows.append({
        "method": "qat_int8_trt",
        "input_bits": bits,
        "top1": r["top1_acc"],
        "top5": r["top5_acc"],
        "lat_ms": r["infer_ms_avg"],
        "lat_std": r.get("infer_ms_std", None),
        "throughput": r.get("throughput_infer_sps", None),
    })

df_trt = pd.DataFrame(rows).sort_values("input_bits", ascending=False).reset_index(drop=True)
df_trt

,method,input_bits,top1,top5,lat_ms,lat_std,throughput
0,qat_int8_trt,8,78.109,93.340,0.701,0.272,1425.575
1,qat_int8_trt,4,78.390,93.561,0.752,0.294,1330.654
2,qat_int8_trt,2,76.439,93.038,0.677,0.263,1476.351
3,qat_int8_trt,1,68.753,89.416,0.710,0.291,1408.608


## Compare: QAT TRT vs QAT PyTorch (fake-quant)

In [ ]:
qat_pytorch_path = PROJECT_ROOT / "resultsv2" / "test_runs" / "qat_test_results.json"
if qat_pytorch_path.exists():
    df_pytorch = pd.read_json(qat_pytorch_path)
    df_pytorch = df_pytorch.rename(columns={
        "top1": "pt_top1", "top5": "pt_top5",
        "lat_ms": "pt_lat_ms", "throughput": "pt_throughput",
    })

    comparison = df_trt.merge(
        df_pytorch[["input_bits", "pt_top1", "pt_top5", "pt_lat_ms", "pt_throughput"]],
        on="input_bits", how="left",
    )
    comparison["top1_delta"] = comparison["top1"] - comparison["pt_top1"]
    comparison["speedup"] = comparison["pt_lat_ms"] / comparison["lat_ms"]
    comparison
else:
    print("No QAT PyTorch results found — run 03_1_qat_test.ipynb first.")
    comparison = df_trt

## Compare: QAT TRT vs PTQ TRT (same precision, different training)

In [ ]:
trt_avg_path = PROJECT_ROOT / "resultsv2" / "val_runs" / "tensorrt_avg_results.json"
if trt_avg_path.exists():
    df_ptq = pd.read_json(trt_avg_path)
    df_ptq_int8 = df_ptq[df_ptq["cfg.model_precision"] == "int8"][
        ["input_bits_trained", "top1_mean", "top5_mean", "infer_ms_mean"]
    ].rename(columns={
        "input_bits_trained": "input_bits",
        "top1_mean": "ptq_trt_top1",
        "top5_mean": "ptq_trt_top5",
        "infer_ms_mean": "ptq_trt_lat_ms",
    })

    vs_ptq = df_trt[["input_bits", "top1", "top5", "lat_ms"]].merge(
        df_ptq_int8, on="input_bits", how="left",
    )
    vs_ptq["qat_vs_ptq_top1"] = vs_ptq["top1"] - vs_ptq["ptq_trt_top1"]
    vs_ptq
else:
    print("No PTQ TRT results found — run 02_tensorrt_inference.ipynb first.")

## Save Results

In [ ]:
out_path = PROJECT_ROOT / "resultsv2" / "test_runs" / "qat_trt_test_results.json"
df_trt.to_json(out_path, orient="records", indent=2)
print(f"Saved to {out_path}")